# 01 — Keluarga Autoencoder: Vanilla AE, Sparse AE, dan VAE

Notebook ini melatih tiga model berbasis autoencoder pada **Fashion-MNIST** dengan split 80/10/10 dan seed 42.

| Model | Encoder | Laten | Decoder | Loss |
|---|---|---|---|---|
| Vanilla AE | 784 → 256 (ReLU) → 64 (linear) | $z\in\mathbb{R}^{64}$ | 64 → 256 (ReLU) → 784 (Sigmoid) | $\mathcal{L}=\frac{1}{D}\lVert x-\hat x\rVert_2^2$ |
| Sparse AE | sama | $z\in\mathbb{R}^{64}$, hidden $h\in\mathbb{R}^{256}$ | sama | $\mathcal{L}=\text{MSE}+\lambda\lVert h\rVert_1,\ \lambda=3\cdot10^{-5}$ |
| VAE | 784 → 256 → $(\mu,\log\sigma^2)\in\mathbb{R}^{64}$ | $z=\mu+\sigma\odot\epsilon$ | sama | $\mathcal{L}=\text{BCE}(x,\hat x)+D_{KL}(q(z|x)\,\|\,\mathcal{N}(0,I))$ |

Encoder 784 → 256 → 64 sengaja dibuat identik dengan MLP Classifier pada notebook 04.

In [ ]:
# ===== Import library =====
import torch                          # framework deep learning utama
import torch.nn.functional as F       # fungsi loss & aktivasi (mse_loss, relu, dll.)
import matplotlib.pyplot as plt       # untuk membuat grafik/visualisasi
from collections import defaultdict   # dict yang otomatis membuat nilai default (dipakai untuk mencatat loss)

# Ambil semua konstanta & helper dari common.py:
# SEED=42, BATCH_SIZE=128, DEVICE (GPU "cuda" jika ada, kalau tidak "cpu"),
# get_loaders, reconstruction_metrics, show_grid, save_json, EpochTimer, CKPT_DIR, FIG_DIR, QUICK
from common import *
# Ambil definisi arsitektur ketiga model dari models.py
from models import VanillaAE, SparseAE, VAE

# Kunci semua sumber keacakan agar hasil eksperimen bisa diulang persis sama
set_seed(SEED)
# Muat Fashion-MNIST (70k gambar) lalu bagi 80% train / 10% val / 10% test
train_loader, val_loader, test_loader = get_loaders(BATCH_SIZE)

# ===== Hyperparameter =====
EPOCHS = 1 if QUICK else 30   # 30 epoch untuk training penuh; 1 epoch jika mode uji cepat (QUICK_RUN=1)
LR = 1e-3                     # learning rate untuk optimizer Adam
LATENT = 64                   # ukuran vektor laten z (kode hasil kompresi)

# Cetak ringkasan: device yang dipakai & jumlah data tiap split
print(f"device={DEVICE} | train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
      f"test={len(test_loader.dataset)} | epochs={EPOCHS}")

## Fungsi loss per model dan training loop generik

In [ ]:
# ===================================================================
# Fungsi "step": menghitung loss untuk SATU batch pada tiap jenis model.
# Return: (loss yang akan di-backprop, dict komponen loss untuk dicatat)
# .detach() = ambil nilainya saja, tidak ikut perhitungan gradien (hanya untuk logging)
# ===================================================================

def step_ae(model, x):
    # Forward pass: model mengembalikan rekonstruksi x_hat, kode laten z, dan aktivasi hidden h
    x_hat, z, h = model(x)
    # Loss = MSE (rata-rata kuadrat selisih piksel asli vs rekonstruksi)
    rec = F.mse_loss(x_hat, x)
    return rec, {"recon_mse": rec.detach()}


def step_sparse(model, x):
    # Forward pass sama seperti Vanilla AE
    x_hat, z, h = model(x)
    # Loss rekonstruksi (MSE)
    rec = F.mse_loss(x_hat, x)
    # Penalti L1 pada aktivasi hidden h: lambda * sum|h| -> memaksa banyak neuron bernilai 0
    pen = model.sparsity_penalty(h)
    # Total loss = MSE + penalti sparsity
    return rec + pen, {"recon_mse": rec.detach(), "l1_penalty": pen.detach()}


def step_vae(model, x):
    # Forward pass VAE: rekonstruksi x_hat, rata-rata mu, dan log-varians logvar dari q(z|x)
    x_hat, mu, logvar = model(x)
    # Loss VAE = BCE (rekonstruksi) + KL divergence (memaksa q(z|x) mendekati N(0, I))
    loss, bce, kl = VAE.loss(x, x_hat, mu, logvar)
    # MSE juga dicatat agar bisa dibandingkan secara adil dengan AE lain
    return loss, {"bce": bce.detach(), "kl": kl.detach(),
                  "recon_mse": F.mse_loss(x_hat, x).detach()}


# ===================================================================
# Training loop generik: dipakai sama untuk ketiga model,
# yang berbeda hanya step_fn (cara menghitung loss)
# ===================================================================
def train_model(model, step_fn, name, epochs=EPOCHS, lr=LR):
    set_seed(SEED)                                      # seed ulang -> inisialisasi tiap model setara/adil
    model = model.to(DEVICE)                            # pindahkan bobot model ke GPU/CPU
    opt = torch.optim.Adam(model.parameters(), lr=lr)   # optimizer Adam untuk update bobot
    hist = defaultdict(list)                            # riwayat loss per epoch, mis. hist["train_loss"] = [...]

    for ep in range(1, epochs + 1):
        # ---------- FASE TRAINING ----------
        model.train()                         # mode training
        sums, n = defaultdict(float), 0       # akumulator jumlah loss & jumlah batch
        with EpochTimer() as tm:              # ukur durasi 1 epoch
            for x, _ in train_loader:         # label (_) diabaikan: autoencoder tidak butuh label
                x = x.to(DEVICE)              # pindahkan batch gambar ke device
                loss, comp = step_fn(model, x)    # 1) forward + hitung loss
                opt.zero_grad(set_to_none=True)   # 2) kosongkan gradien dari langkah sebelumnya
                loss.backward()                   # 3) backpropagation: hitung gradien
                opt.step()                        # 4) update bobot berdasarkan gradien
                # akumulasi loss untuk dihitung rata-ratanya di akhir epoch
                sums["loss"] += loss.detach()
                for k, v in comp.items():
                    sums[k] += v
                n += 1
        # simpan waktu epoch dan rata-rata loss training per batch
        hist["epoch_time"].append(tm.seconds)
        for k, v in sums.items():
            hist["train_" + k].append(float(v) / n)

        # ---------- FASE VALIDASI ----------
        model.eval()                          # mode evaluasi
        vs, vn = defaultdict(float), 0
        with torch.no_grad():                 # tanpa gradien -> lebih cepat & hemat memori, bobot tidak berubah
            for x, _ in val_loader:
                loss, comp = step_fn(model, x.to(DEVICE))   # hanya forward + hitung loss
                vs["loss"] += loss
                for k, v in comp.items():
                    vs[k] += v
                vn += 1
        # simpan rata-rata loss validasi
        for k, v in vs.items():
            hist["val_" + k].append(float(v) / vn)

        # cetak progres pada epoch 1, kelipatan 5, dan epoch terakhir
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{name}] epoch {ep:02d} | train {hist['train_loss'][-1]:.5f} "
                  f"| val {hist['val_loss'][-1]:.5f} | val MSE {hist['val_recon_mse'][-1]:.5f} "
                  f"| {tm.seconds:.1f}s")

    # Simpan bobot model terlatih ke checkpoints/<name>.pt
    torch.save(model.state_dict(), os.path.join(CKPT_DIR, f"{name}.pt"))
    # Simpan riwayat training ke results/history_<name>.json (untuk plot kurva loss)
    save_json(dict(hist), f"history_{name}.json")
    return model, hist

## Training

In [ ]:
# Latih Vanilla AE (784 -> 256 -> 64 -> 256 -> 784) dengan loss MSE
vanilla_ae, hist_ae = train_model(VanillaAE(LATENT), step_ae, "vanilla_ae")

In [ ]:
# Latih Sparse AE (arsitektur sama dengan Vanilla AE) dengan loss MSE + penalti L1 pada hidden layer
sparse_ae, hist_sae = train_model(SparseAE(LATENT), step_sparse, "sparse_ae")

In [ ]:
# Latih VAE: encoder menghasilkan (mu, logvar), z diambil via reparameterization trick,
# loss = BCE + KL divergence
vae, hist_vae = train_model(VAE(LATENT), step_vae, "vae")

## Evaluasi rekonstruksi pada Test Set (MSE, PSNR, SSIM)
VAE dievaluasi memakai rekonstruksi deterministik $\hat x = g(\mu(x))$.

In [ ]:
# Set semua model ke mode evaluasi sebelum diuji
for m in (vanilla_ae, sparse_ae, vae):
    m.eval()

# Fungsi rekonstruksi per model: input gambar x -> output rekonstruksi x_hat
recon_fns = {
    "Vanilla AE": lambda x: vanilla_ae(x)[0],             # [0] = ambil x_hat dari (x_hat, z, h)
    "Sparse AE": lambda x: sparse_ae(x)[0],
    "VAE": lambda x: vae.decode(vae.encode(x)[0]),        # pakai mu langsung (tanpa noise) -> deterministik
}

# Hitung MSE, PSNR, SSIM pada Test Set untuk tiap model
test_metrics = {k: reconstruction_metrics(f, test_loader) for k, f in recon_fns.items()}

# Tampilkan hasil: MSE kecil = baik, PSNR besar = baik, SSIM mendekati 1 = baik
for k, v in test_metrics.items():
    print(f"{k:11s} MSE={v['mse']:.5f}  PSNR={v['psnr']:.2f} dB  SSIM={v['ssim']:.4f}")

# Simpan metrik ke results/recon_metrics_autoencoders.json
save_json(test_metrics, "recon_metrics_autoencoders.json")

## Analisis struktur fitur hidden layer: Vanilla AE vs Sparse AE
Penalti $\lambda\lVert h\rVert_1$ mendorong sebagian besar unit hidden bernilai nol, sehingga setiap citra hanya diwakili
oleh sedikit unit aktif.

In [ ]:
@torch.no_grad()   # tidak perlu gradien karena hanya mengambil aktivasi
def hidden_acts(model):
    # Ambil aktivasi hidden h (256 neuron) untuk seluruh Test Set
    # encode(..., return_hidden=True) mengembalikan (z, h) -> [1] = h
    # Hasil akhir: matriks (jumlah_gambar_test, 256)
    return torch.cat([model.encode(x.to(DEVICE), return_hidden=True)[1].cpu() for x, _ in test_loader])

# Aktivasi hidden untuk Vanilla AE dan Sparse AE
H_ae, H_sae = hidden_acts(vanilla_ae), hidden_acts(sparse_ae)

sparsity = {}
for name, H in (("Vanilla AE", H_ae), ("Sparse AE", H_sae)):
    sparsity[name] = {
        # persentase aktivasi yang bernilai ~0 (neuron "mati" untuk gambar tsb)
        "pct_zero_activations": (H <= 1e-6).float().mean().item() * 100,
        # rata-rata jumlah neuron yang aktif (>0) per gambar
        "mean_active_units_per_image": (H > 1e-6).float().sum(1).mean().item(),
        # rata-rata besar aktivasi
        "mean_abs_activation": H.abs().mean().item(),
        # jumlah neuron yang TIDAK PERNAH aktif di seluruh Test Set
        "dead_units_never_active": int(((H > 1e-6).sum(0) == 0).sum()),
        # Hoyer sparsity (0 = padat, 1 = sangat jarang)
        # rumus: (sqrt(n) - L1/L2) / (sqrt(n) - 1), dihitung per gambar lalu dirata-rata
        "hoyer_sparsity": (((H.shape[1] ** 0.5) - H.abs().sum(1) / H.norm(dim=1).clamp_min(1e-8))
                           / ((H.shape[1] ** 0.5) - 1)).mean().item(),
    }
    print(name, {k: round(v, 3) for k, v in sparsity[name].items()})

# Unit VAE aktif: dimensi laten dengan KL rata-rata > 0.01 nat
with torch.no_grad():
    # encode seluruh Test Set -> kumpulan (mu, logvar) per batch
    mus, lvs = zip(*[vae.encode(x.to(DEVICE)) for x, _ in test_loader])
    mu, lv = torch.cat(mus), torch.cat(lvs)   # gabungkan jadi (N, 64)
    # KL per dimensi laten, dirata-rata atas semua gambar -> vektor 64 angka
    kl_dim = (-0.5 * (1 + lv - mu.pow(2) - lv.exp())).mean(0).cpu()

# Dimensi dengan KL <= 0.01 dianggap tidak terpakai (posterior collapse): isinya cuma noise N(0,1)
sparsity["VAE"] = {"active_latent_dims_kl>0.01": int((kl_dim > 0.01).sum()),
                   "kl_per_dim": kl_dim.tolist()}
print("VAE active latent dims:", sparsity["VAE"]["active_latent_dims_kl>0.01"], "/", LATENT)

# Simpan semua statistik sparsity ke results/sparsity_stats.json
save_json(sparsity, "sparsity_stats.json")

In [ ]:
# Buat figure dengan 3 panel: [histogram | filter Vanilla AE | filter Sparse AE]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.1, 1, 1]})

# ---------- Panel 1: histogram distribusi aktivasi hidden ----------
# 60 bin dari 0 sampai nilai aktivasi maksimum (dipakai bersama agar kedua histogram sebanding)
bins = torch.linspace(0, float(max(H_ae.max(), H_sae.max())), 60)
# log=True -> sumbu y skala log, supaya bin dengan frekuensi kecil tetap terlihat
axes[0].hist(H_ae.flatten().numpy(), bins=bins.numpy(), alpha=0.6, label="Vanilla AE", log=True)
axes[0].hist(H_sae.flatten().numpy(), bins=bins.numpy(), alpha=0.6, label="Sparse AE", log=True)
axes[0].set_title("Distribusi aktivasi hidden $h$ (Test Set, skala log)")
axes[0].set_xlabel("nilai aktivasi"); axes[0].legend()


def filters(model):
    """Baris W1 (256 x 784) sebagai 256 filter 28x28."""
    # Tiap baris bobot fc1 (784 angka) diubah bentuknya menjadi gambar 28x28
    # -> menunjukkan pola piksel yang "dicari" oleh tiap neuron hidden
    return model.fc1.weight.detach().cpu().view(-1, 1, 28, 28)


# ---------- Panel 2 & 3: 36 filter encoder paling aktif ----------
for ax, (name, model, H) in zip(axes[1:], (("Vanilla AE", vanilla_ae, H_ae), ("Sparse AE", sparse_ae, H_sae))):
    W = filters(model)   # (256, 1, 28, 28)
    # urutkan berdasarkan frekuensi aktivasi agar filter yang paling sering dipakai tampil
    order = (H > 1e-6).float().mean(0).argsort(descending=True)[:36]
    Wk = W[order]        # ambil 36 filter teratas
    # Normalisasi min-max per filter ke rentang [0, 1] agar bisa ditampilkan sebagai gambar
    Wk = (Wk - Wk.amin(dim=(1, 2, 3), keepdim=True)) / (Wk.amax(dim=(1, 2, 3), keepdim=True)
                                                        - Wk.amin(dim=(1, 2, 3), keepdim=True))
    # Tampilkan dalam grid 6x6
    show_grid(Wk, nrow=6, ax=ax, title=f"{name}: 36 filter encoder $W_1$ paling aktif")

plt.tight_layout()
# Simpan gambar ke folder figures/
plt.savefig(os.path.join(FIG_DIR, "figS1_sparse_vs_vanilla_features.png"), dpi=200, bbox_inches="tight")
plt.show()

## Pratinjau rekonstruksi dan sampel VAE

In [ ]:
# Ambil 1 batch pertama dari Test Set, lalu pakai 8 gambar saja
x, _ = next(iter(test_loader))
x = x[:8].to(DEVICE)

with torch.no_grad():
    # Baris 1: gambar asli (GT); baris 2-4: rekonstruksi Vanilla AE, Sparse AE, VAE
    rows = [x] + [f(x) for f in recon_fns.values()]
    set_seed(SEED)   # agar sampel acak VAE selalu sama setiap dijalankan
    # GENERASI: ambil 16 vektor z acak dari N(0, I), lalu decode menjadi gambar BARU
    # (hanya VAE yang bisa begini karena ruang latennya dilatih mendekati N(0, I))
    samples = vae.decode(torch.randn(16, LATENT, device=DEVICE))

# Tampilkan 2 panel: kiri = perbandingan rekonstruksi, kanan = sampel baru VAE
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
show_grid(torch.cat(rows), nrow=8, ax=ax[0], title="GT / Vanilla AE / Sparse AE / VAE")
show_grid(samples, nrow=4, ax=ax[1], title="Sampel VAE dari $z\\sim\\mathcal{N}(0,I)$")
plt.show()